# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants and clients
load_dotenv(override=True)
MODEL_GPT = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
MODEL_LLAMA = os.getenv('OLLAMA_MODEL', 'llama3.2')
MODEL_LLAMA_CLOUD = os.getenv('OLLAMA_CLOUD_MODEL', 'gpt-oss:20b-cloud')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')

In [ ]:
# set up environment
api_key = os.getenv('OPENAI_API_KEY')
api_key_ollama = os.getenv('OLLAMA_API_KEY') or 'ollama'

# sanity check OPENAI keys
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print('API key looks good so far')
else:
    print('There might be a problem with your API key? Please visit the troubleshooting notebook!')

# setup for OpenAI
openai = OpenAI()

# setup for Ollama clients (same local endpoint, different model names)
if api_key_ollama == 'ollama':
    print('Using default local Ollama API key')
else:
    print('Using configured Ollama API key')

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key_ollama)
ollama_cloud = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key_ollama)

In [ ]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

system_prompt = """
You are a clear and patient coding assistant. When given a piece of code, you will:
1. Explain what the code does, step by step.
2. Describe why the code works the way it does.
3. Provide a simple, practical example of how the code could be used.
Your explanations should be easy to follow for someone new to programming and avoid unnecessary technical jargon.
"""

user_prompt = f"""
Please explain the following code clearly and give an example of its use:
```
{question}
```
"""

In [ ]:
# function to stream response and update display
def stream_response(client, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )
    
    display_handle = display(Markdown(""), display_id=True)
    full_response = ""
    for chunk in response:
        response_text = chunk.choices[0].delta.content or ''
        full_response += response_text
        update_display(Markdown(full_response), display_id=display_handle.display_id)

In [ ]:
stream_response(openai, MODEL_GPT)

In [ ]:
stream_response(ollama, MODEL_LLAMA)

In [ ]:
stream_response(ollama_cloud, MODEL_LLAMA_CLOUD)